# **Inferencia y Validación de Hipótesis**

### Decisiones metodológicas

- **α = 0.05** con **corrección Holm-Bonferroni** para familias de comparaciones múltiples (post-hoc por tipologías, por zonas de riesgo, etc.).
- **Filtros previos**: `geo_ok = True`, **excluir super-premium** (`Precio_m2_USD > p99`), y **excluir `confianza_renta = baja`** en tests que involucran `Rentabilidad` o `Payback`.
- **Tests paramétricos y no paramétricos en paralelo**: dado que el EDA mostró fuerte asimetría, los no paramétricos (Mann-Whitney, Kruskal-Wallis, Spearman) son los principales para decisión; los paramétricos (T-Student, ANOVA, Pearson) se reportan con sus diagnósticos de supuestos (Shapiro-Wilk para normalidad, Levene para homocedasticidad).



## 1. Setup

Imports y helpers. La función `reportar_test` estandariza la salida de cada prueba para que la lectura sea homogénea en todo el notebook.

In [ ]:
# Instalación: si statsmodels no está instalado
# !pip install -q statsmodels

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.multicomp import pairwise_tukeyhsd

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 5)
pd.set_option('display.max_columns', 80)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

ALPHA = 0.05

def reportar_test(nombre, h0, h1, stat_label, stat, p_val, alpha=ALPHA, p_adj=None, extra=None):
    """Estandariza el reporte de un test estadístico."""
    p_use = p_adj if p_adj is not None else p_val
    decision = 'RECHAZAMOS H0' if p_use < alpha else 'NO rechazamos H0'
    print(f'--- {nombre} ---')
    print(f'  H0: {h0}')
    print(f'  H1: {h1}')
    print(f'  {stat_label}: {stat:,.4f}   |   p-valor: {p_val:.4g}')
    if p_adj is not None:
        print(f'  p-valor ajustado (Holm): {p_adj:.4g}')
    if extra:
        for k, v in extra.items():
            print(f'  {k}: {v}')
    print(f'  α = {alpha} → {decision}\n')
    return {'test': nombre, 'estadistico': stat, 'p_valor': p_val,
            'p_ajustado': p_adj, 'decision': decision}

resultados = []  # acumulador para la síntesis final

## 2. Carga y filtros

Aplicamos los tres filtros acordados:
1. `geo_ok = True` (mismo criterio que el EDA).
2. Exclusión de super-premium (`Precio_m2_USD > p99`). Documentamos cuántas filas se descartan.
3. Exclusión de `confianza_renta = 'baja'` **solo en los tests que involucran Rentabilidad o Payback** (`df_v_conf`). Para los tests que solo usan precio o superficie usamos `df_v`.

In [ ]:
URL_ALQ    = 'https://raw.githubusercontent.com/kiaranatale/Trabajo-Practico-Real-State-Analytics/refs/heads/main/Datos2/dataset_alquileres_features.csv'
URL_VENTAS = 'https://raw.githubusercontent.com/kiaranatale/Trabajo-Practico-Real-State-Analytics/refs/heads/main/Datos2/dataset_ventas_features.csv'

df_alq_raw    = pd.read_csv(URL_ALQ)
df_ventas_raw = pd.read_csv(URL_VENTAS)

print(f'Ventas raw:     {df_ventas_raw.shape}')
print(f'Alquileres raw: {df_alq_raw.shape}')

# Filtro 1: geo_ok
df_v_geo = df_ventas_raw[df_ventas_raw['geo_ok'] == True].copy().reset_index(drop=True)
df_a_geo = df_alq_raw   [df_alq_raw   ['geo_ok'] == True].copy().reset_index(drop=True)

# Tipos numéricos
num_cols = ['Precio_USD','Precio_ajustado_USD','Precio_m2_USD','Sup_Cubierta_m2',
            'Antiguedad','Ambientes','Comuna','Renta_estimada_mensual_USD',
            'Renta_m2_USD','Renta_largo_mensual_USD','Rentabilidad_bruta_anual',
            'Rentabilidad_neta_anual','Payback_anios','Yield_gap_vs_bonos',
            'Distancia_subte_km','Distancia_parque_km','Indice_amenities',
            'Tasa_delitos_comuna']
for df in (df_v_geo, df_a_geo):
    for c in num_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')

# Filtro 2: super-premium (>p99 Precio_m2_USD)
p99_pm2_v = df_v_geo['Precio_m2_USD'].quantile(0.99)
print(f'\nUmbral p99 Precio/m² ventas: {p99_pm2_v:,.0f} USD/m²')
df_v = df_v_geo[df_v_geo['Precio_m2_USD'] <= p99_pm2_v].copy().reset_index(drop=True)
print(f'Ventas: {len(df_v_geo):,} → {len(df_v):,}  (descartados super-premium: {len(df_v_geo)-len(df_v):,})')

p99_rm2_a = df_a_geo['Renta_m2_USD'].quantile(0.99)
df_a = df_a_geo[df_a_geo['Renta_m2_USD'] <= p99_rm2_a].copy().reset_index(drop=True)
print(f'Alquileres: {len(df_a_geo):,} → {len(df_a):,}  (descartados super-premium por Renta/m²: {len(df_a_geo)-len(df_a):,})')

# Filtro 3: confianza_renta != baja (solo para tests de rentabilidad)
df_v_conf = df_v[df_v['confianza_renta'] != 'baja'].copy().reset_index(drop=True)
print(f'\nVentas con confianza_renta ≥ media: {len(df_v_conf):,} ({len(df_v_conf)/len(df_v)*100:.1f}% de df_v)')
print(f'  → df_v_conf se usará en tests de Rentabilidad y Payback')

Ventas raw:     (3868, 91)
Alquileres raw: (3645, 80)

Umbral p99 Precio/m² ventas: 5,771 USD/m²
Ventas: 3,493 → 3,456  (descartados super-premium: 37)
Alquileres: 3,348 → 3,314  (descartados super-premium por Renta/m²: 34)

Ventas con confianza_renta ≥ media: 3,140 (90.9% de df_v)
  → df_v_conf se usará en tests de Rentabilidad y Payback


## 3. Derivación de `tipo_alquiler` (temporal vs tradicional)

El dataset de alquileres mezcla **alquiler tradicional de largo plazo** y **alquiler temporal**. Los podemos distinguir por patrón en la URL del aviso: `alquiler-temporal` aparece en los temporales (típicamente Argenprop). Esta es la variable necesaria para validar H6.

> **Caveat importante**: los precios publicados en alquiler temporal pueden ser mensuales (estadía mensual amueblada) o por noche. Si son mensuales, son comparables con tradicional; si son por noche, la comparación queda escalada. Inspeccionamos las medianas y la forma de la distribución para decidir.

In [ ]:

print('Mediana de Renta_largo_mensual_USD por tipo_alquiler:')
print(df_a.groupby('tipo_alquiler')['Renta_largo_mensual_USD'].agg(['count','median','mean']).round(0))
print()
print('Mediana de Renta_m2_USD por tipo_alquiler (más comparable):')
print(df_a.groupby('tipo_alquiler')['Renta_m2_USD'].agg(['count','median','mean']).round(2))

Mediana de Renta_largo_mensual_USD por tipo_alquiler:
               count   median       mean
tipo_alquiler                           
desconocido      236 641.0000 1,074.0000
temporal         128 800.0000   917.0000
tradicional     2950 506.0000   706.0000

Mediana de Renta_m2_USD por tipo_alquiler (más comparable):
               count  median    mean
tipo_alquiler                       
desconocido      236 13.3300 13.9300
temporal         128 16.7700 17.5100
tradicional     2950 12.2100 12.8200


In [ ]:
temp = df_a.loc[df_a['tipo_alquiler']=='temporal',   'Renta_m2_USD'].dropna()
trad = df_a.loc[df_a['tipo_alquiler']=='tradicional','Renta_m2_USD'].dropna()


# Diagnósticos previos
_, p_lev = stats.levene(temp, trad, center='median')
print(f'  Levene p={p_lev:.3g} → varianzas {"distintas" if p_lev<0.05 else "compatibles"}  '
      f'(uso Welch para el T por el desbalance 10:1)')

# Mann-Whitney + tamaño de efecto (rank-biserial = 1 - 2U/(n1·n2))
u, p_mw = stats.mannwhitneyu(temp, trad, alternative='two-sided')
r_rb = 1 - (2*u) / (len(temp)*len(trad))
interp_rb = 'despreciable' if abs(r_rb)<0.1 else 'chico' if abs(r_rb)<0.3 else 'mediano' if abs(r_rb)<0.5 else 'grande'

# T-Student con Welch
t, p_t = stats.ttest_ind(temp, trad, equal_var=False)

# Bootstrap del diferencial de medianas (más robusto al desbalance)
rng = np.random.default_rng(1)
diffs = []
for _ in range(2000):
    bt = rng.choice(temp.values, size=len(temp), replace=True)
    bs = rng.choice(trad.values, size=len(trad), replace=True)
    diffs.append(np.median(bt) - np.median(bs))
ic95 = np.percentile(diffs, [2.5, 97.5])
print(f'  Diferencia mediana temp − trad: {temp.median()-trad.median():.2f}  '
      f'IC95% bootstrap: [{ic95[0]:.2f}, {ic95[1]:.2f}]')

resultados.append(reportar_test(
    'H6 · Mann-Whitney: Renta/m² (temporal vs tradicional)',
    h0='medianas iguales',
    h1='medianas distintas (esperamos temporal > tradicional)',
    stat_label='U', stat=u, p_val=p_mw,
    extra={'Tamaño efecto (rank-biserial)': f'r={r_rb:.3f} ({interp_rb})',
            'IC95% bootstrap dif. medianas': f'[{ic95[0]:.2f}, {ic95[1]:.2f}] USD/m²/mes',
            'Caveat 1': 'desbalance 10:1 — un p chico puede no implicar diferencia económica grande',
            'Caveat 2': 'no controla por zona / portal / si el temporal es mensual o por noche'}))
resultados.append(reportar_test(
    'H6 · T-Welch    : Renta/m² (temporal vs tradicional)',
    h0='medias iguales', h1='medias distintas',
    stat_label='t', stat=t, p_val=p_t))

  Levene p=1.34e-08 → varianzas distintas  (uso Welch para el T por el desbalance 10:1)
  Diferencia mediana temp − trad: 4.57  IC95% bootstrap: [3.42, 6.21]
--- H6 · Mann-Whitney: Renta/m² (temporal vs tradicional) ---
  H0: medianas iguales
  H1: medianas distintas (esperamos temporal > tradicional)
  U: 285,990.0000   |   p-valor: 5.398e-23
  Tamaño efecto (rank-biserial): r=-0.515 (grande)
  IC95% bootstrap dif. medianas: [3.42, 6.21] USD/m²/mes
  Caveat 1: desbalance 10:1 — un p chico puede no implicar diferencia económica grande
  Caveat 2: no controla por zona / portal / si el temporal es mensual o por noche
  α = 0.05 → RECHAZAMOS H0

--- H6 · T-Welch    : Renta/m² (temporal vs tradicional) ---
  H0: medias iguales
  H1: medias distintas
  t: 9.1639   |   p-valor: 8.131e-16
  α = 0.05 → RECHAZAMOS H0



Como era esperable, las varianzas entre temporal y tradicional son diferentes. Las medianas son distintas, con un p valor pequeño para mann withney.Como era de esperarse, los alquileres temporales tienen rentas más altas. Welch confirma.

El temporal cobra entre 3.42 y 6.21 USD/m²/mes más que el tradicional, con 95% de confianza. Para un departamento típico de 50 m²: entre +171 y +311 USD/mes de renta extra. Para un 2A de 45 m² en zona céntrica: +154 a +279 USD/mes.

H6 se valida estadísticamente con efecto grande (p << 0.05, r = 0.51, IC95% lejos del cero). La prima de precio publicado es al menos 3.4 USD/m²/mes. Sin embargo, se hubiera preferido tener tamaños de muestras parecidos, y uno mas grande para alquileres temporales.

No vamos a desagregar por comunas porque son muy pocos alquileres temporales.

## 4. H1 — Cercanía al subte se capitaliza en precio, no en rentabilidad

Una propiedad cerca de subte es más demandada → precio sube. Como inversión, el ingreso por alquiler crece proporcionalmente al precio. Resultado: rentabilidad (renta/precio) se mantiene similar.

Descomponemos en **dos sub-hipótesis testeables**:

### H1.a — Precio/m² depende negativamente de la distancia al subte

- **H0**: ρ(Distancia_subte, Precio_m2) = 0
- **H1**: ρ(Distancia_subte, Precio_m2) ≠ 0

### H1.b — Rentabilidad bruta NO depende de la distancia al subte

- **H0**: ρ(Distancia_subte, Rentabilidad_bruta) = 0
- **H1**: ρ(Distancia_subte, Rentabilidad_bruta) ≠ 0



**Lectura de H1:**

- **H1.a (precio vs distancia subte)**: si el ρ de Spearman es **negativo y p < α**, rechazamos H0 → hay capitalización en precio. ✅ Coherente con H1.
- **H1.b (rentabilidad vs distancia subte)**: si el ρ es **cercano a cero y p ≥ α**, NO rechazamos H0 → la rentabilidad no depende de la distancia al subte. ✅ Coherente con H1 (precio sube, renta sube en igual proporción).
- Cuando rechazamos H1.b por p < α, hay que mirar el **tamaño del efecto** (|ρ|). Con n grande, ρ = 0.05 puede ser significativo pero económicamente irrelevante. Por eso reportamos también |ρ|.

In [ ]:
# ----- H1.a: Distancia_subte vs Precio_m2 (Spearman + Pearson) -----
d = df_v[['Distancia_subte_km','Precio_m2_USD']].dropna()
rho_s, p_s = stats.spearmanr(d['Distancia_subte_km'], d['Precio_m2_USD'])
rho_p, p_p = stats.pearsonr (d['Distancia_subte_km'], d['Precio_m2_USD'])

resultados.append(reportar_test(
    'H1.a · Spearman: Distancia_subte vs Precio/m²',
    h0='ρ = 0 (sin asociación monótona)',
    h1='ρ ≠ 0 (esperamos ρ < 0 por capitalización)',
    stat_label='ρ Spearman', stat=rho_s, p_val=p_s,
    extra={'n': len(d), 'Dirección': 'negativa' if rho_s<0 else 'positiva'}))
resultados.append(reportar_test(
    'H1.a · Pearson  : Distancia_subte vs Precio/m² (paramétrico)',
    h0='r = 0 (sin asociación lineal)',
    h1='r ≠ 0',
    stat_label='r Pearson', stat=rho_p, p_val=p_p,
    extra={'n': len(d), 'Supuesto': 'normalidad bivariada (probablemente violado en datos crudos)'}))

--- H1.a · Spearman: Distancia_subte vs Precio/m² ---
  H0: ρ = 0 (sin asociación monótona)
  H1: ρ ≠ 0 (esperamos ρ < 0 por capitalización)
  ρ Spearman: 0.0044   |   p-valor: 0.7946
  n: 3456
  Dirección: positiva
  α = 0.05 → NO rechazamos H0

--- H1.a · Pearson  : Distancia_subte vs Precio/m² (paramétrico) ---
  H0: r = 0 (sin asociación lineal)
  H1: r ≠ 0
  r Pearson: -0.0835   |   p-valor: 8.792e-07
  n: 3456
  Supuesto: normalidad bivariada (probablemente violado en datos crudos)
  α = 0.05 → RECHAZAMOS H0



El efecto del subte sobre el precio existe pero es tan chico que se diluye en la variabilidad general — Pearson lo capta solo por el n grande (3.456 observaciones), no por magnitud real (un |r| de 0.08 explica menos del 1% de la varianza).

H1.a queda no apoyada de forma robusta: la cercanía al subte no es un driver fuerte del precio/m² en CABA, probablemente porque casi toda la ciudad está dentro de 1.5 km de alguna estación y la variable pierde poder discriminante.

In [ ]:
# ----- H1.b: Distancia_subte vs Rentabilidad_bruta (Spearman) -----
d = df_v_conf[['Distancia_subte_km','Rentabilidad_bruta_anual']].dropna()
rho_s, p_s = stats.spearmanr(d['Distancia_subte_km'], d['Rentabilidad_bruta_anual'])
rho_p, p_p = stats.pearsonr (d['Distancia_subte_km'], d['Rentabilidad_bruta_anual'])

resultados.append(reportar_test(
    'H1.b · Spearman: Distancia_subte vs Rentabilidad bruta',
    h0='ρ = 0 (sin asociación) — APOYAR H0 confirma la teoría',
    h1='ρ ≠ 0',
    stat_label='ρ Spearman', stat=rho_s, p_val=p_s,
    extra={'n': len(d),
           'Tamaño efecto': f'|ρ|={abs(rho_s):.3f} ({"despreciable" if abs(rho_s)<0.1 else "débil" if abs(rho_s)<0.3 else "moderado o fuerte"})'}))
resultados.append(reportar_test(
    'H1.b · Pearson  : Distancia_subte vs Rentabilidad bruta',
    h0='r = 0', h1='r ≠ 0',
    stat_label='r Pearson', stat=rho_p, p_val=p_p,
    extra={'n': len(d)}))

--- H1.b · Spearman: Distancia_subte vs Rentabilidad bruta ---
  H0: ρ = 0 (sin asociación) — APOYAR H0 confirma la teoría
  H1: ρ ≠ 0
  ρ Spearman: -0.0494   |   p-valor: 0.005664
  n: 3140
  Tamaño efecto: |ρ|=0.049 (despreciable)
  α = 0.05 → RECHAZAMOS H0

--- H1.b · Pearson  : Distancia_subte vs Rentabilidad bruta ---
  H0: r = 0
  H1: r ≠ 0
  r Pearson: -0.0463   |   p-valor: 0.009409
  n: 3140
  α = 0.05 → RECHAZAMOS H0



Ambos tests rechazan H0 (Spearman ρ = −0.05, p = 0.006; Pearson r = −0.05, p = 0.009), pero el tamaño de efecto es despreciable (|ρ| < 0.05, explica menos del 0.3% de la varianza) — el rechazo es puro artefacto del n grande (3.140 observaciones), no evidencia de relación económica. La dirección es además opuesta a la que esperaríamos si el subte penalizara la rentabilidad: el signo negativo dice que más distancia se asocia a menor rentabilidad, no mayor, descartando el mecanismo "subte se paga en precio sin recuperarse en renta".

H1.b queda apoyada en lo sustantivo aunque estadísticamente rechacemos H0: la rentabilidad bruta es esencialmente independiente de la distancia al subte, confirmando que el subte no genera yield diferencial.

In [ ]:
# ----- H1: comparación cerca (≤ 0.5 km) vs lejos (> 1 km) -----
def grupo_subte(d):
    if pd.isna(d):       return None
    if d <= 0.5:         return 'cerca'
    if d >  1.0:         return 'lejos'
    return None  # intermedio se excluye para hacer contraste limpio

for col_y, df_use, label in [
    ('Precio_m2_USD',           df_v,      'Precio/m²'),
    ('Rentabilidad_bruta_anual', df_v_conf, 'Rentabilidad bruta'),
]:
    df_use = df_use.copy()
    df_use['_g'] = df_use['Distancia_subte_km'].apply(grupo_subte)
    cerca = df_use.loc[df_use['_g']=='cerca', col_y].dropna()
    lejos = df_use.loc[df_use['_g']=='lejos', col_y].dropna()

    # Diagnósticos de supuestos para el paramétrico
    # Shapiro requiere n ≤ 5000
    sample_c = cerca.sample(min(len(cerca), 4900), random_state=1)
    sample_l = lejos.sample(min(len(lejos), 4900), random_state=1)
    _, p_sh_c = stats.shapiro(sample_c)
    _, p_sh_l = stats.shapiro(sample_l)
    _, p_lev  = stats.levene(cerca, lejos, center='median')

    u, p_mw = stats.mannwhitneyu(cerca, lejos, alternative='two-sided')
    t, p_t  = stats.ttest_ind  (cerca, lejos, equal_var=(p_lev > 0.05))

    print(f'╔══ {label} cerca vs lejos del subte ══╗')
    print(f'  n cerca: {len(cerca):,}  | mediana: {cerca.median():,.3f}')
    print(f'  n lejos: {len(lejos):,}  | mediana: {lejos.median():,.3f}')
    print(f'  Shapiro p (cerca): {p_sh_c:.3g}  | (lejos): {p_sh_l:.3g}  ← normalidad: {"se viola" if min(p_sh_c,p_sh_l)<0.05 else "OK"}')
    print(f'  Levene p: {p_lev:.3g}  ← homocedasticidad: {"se viola" if p_lev<0.05 else "OK"}')
    resultados.append(reportar_test(
        f'H1 · Mann-Whitney {label} (cerca vs lejos)',
        h0=f'mediana({label}|cerca) = mediana({label}|lejos)',
        h1='medianas distintas',
        stat_label='U', stat=u, p_val=p_mw))
    resultados.append(reportar_test(
        f'H1 · T-Student {label} (cerca vs lejos)',
        h0=f'μ({label}|cerca) = μ({label}|lejos)',
        h1='medias distintas',
        stat_label='t', stat=t, p_val=p_t))

╔══ Precio/m² cerca vs lejos del subte ══╗
  n cerca: 1,100  | mediana: 2,378.511
  n lejos: 1,354  | mediana: 2,465.692
  Shapiro p (cerca): 4.29e-08  | (lejos): 5.38e-11  ← normalidad: se viola
  Levene p: 0.125  ← homocedasticidad: OK
--- H1 · Mann-Whitney Precio/m² (cerca vs lejos) ---
  H0: mediana(Precio/m²|cerca) = mediana(Precio/m²|lejos)
  H1: medianas distintas
  U: 702,717.5000   |   p-valor: 0.01617
  α = 0.05 → RECHAZAMOS H0

--- H1 · T-Student Precio/m² (cerca vs lejos) ---
  H0: μ(Precio/m²|cerca) = μ(Precio/m²|lejos)
  H1: medias distintas
  t: -2.2460   |   p-valor: 0.02479
  α = 0.05 → RECHAZAMOS H0

╔══ Rentabilidad bruta cerca vs lejos del subte ══╗
  n cerca: 1,038  | mediana: 0.064
  n lejos: 1,154  | mediana: 0.060
  Shapiro p (cerca): 1.78e-51  | (lejos): 3.14e-41  ← normalidad: se viola
  Levene p: 0.000575  ← homocedasticidad: se viola
--- H1 · Mann-Whitney Rentabilidad bruta (cerca vs lejos) ---
  H0: mediana(Rentabilidad bruta|cerca) = mediana(Rentabilidad b

Los cuatro tests rechazan H0, pero hay que leer los signos y magnitudes, no solo los p-valores. Para precio/m², las propiedades lejos del subte tienen mediana mayor (2.466 vs 2.378 USD/m²) — es decir, el efecto va en dirección contraria a la esperada por capitalización del subte, con una diferencia chiquita de ~3.7%; el rechazo de H0 confirma que las distribuciones difieren pero refuta la H1.a en su versión "cerca al subte = más caro". Para rentabilidad bruta, cerca tiene mediana 6.4% y lejos 6.0% — diferencia de 40 puntos básicos, estadísticamente significativa pero económicamente marginal, y otra vez en dirección opuesta a la teoría (la teoría predice rentabilidad igual o levemente mayor lejos del subte, no menor).

El cuadro completo de H1 sugiere que en CABA el subte no es un factor estructural de precios ni de yield: las diferencias existen pero son pequeñas, ruidosas y direccionalmente inconsistentes con el supuesto del README, probablemente porque la cobertura geográfica del subte es heterogénea (norte saturado, sur sin red) y los barrios "lejos" del subte incluyen tanto zonas premium como Belgrano R o Núñez como zonas suburbanas baratas, lavando el efecto.

## 5. H2 — Tamaño (tipología) genera rentabilidad diferencial

**Lógica causal:** propiedades pequeñas tienen menor precio absoluto pero renta casi tan alta → ratio renta/precio es superior.

### Test global

- **H0**: la distribución (mediana / media) de la rentabilidad bruta es la misma entre las 4 tipologías (monoamb, 2A, 3A, 4+A).
- **H1**: al menos una tipología difiere.

### Post-hoc (si rechazamos H0)

Identificamos *qué pares* difieren con comparaciones pareadas **ajustadas por Holm** (no paramétrico: Mann-Whitney pairwise; paramétrico: Tukey HSD).

In [ ]:
# ----- H2: Kruskal-Wallis + ANOVA sobre Rentabilidad por Tipología -----
TIPS = ['monoamb','2A','3A','4+A']
grupos = [df_v_conf.loc[df_v_conf['Tipologia']==t, 'Rentabilidad_bruta_anual'].dropna() for t in TIPS]
for t, g in zip(TIPS, grupos):
    print(f'  Tipologia {t:<7} → n={len(g):,}  mediana={g.median():.4f}  media={g.mean():.4f}')

H, p_kw = stats.kruskal(*grupos)
F, p_an = stats.f_oneway(*grupos)

# Test de homocedasticidad para validar supuesto de ANOVA
_, p_lev = stats.levene(*grupos, center='median')
print(f'\nLevene (homocedasticidad): p={p_lev:.3g} → {"se viola" if p_lev<0.05 else "OK"}')

resultados.append(reportar_test(
    'H2 · Kruskal-Wallis: Rentabilidad bruta por Tipología',
    h0='Distribuciones idénticas entre tipologías',
    h1='Al menos una difiere',
    stat_label='H', stat=H, p_val=p_kw))
resultados.append(reportar_test(
    'H2 · ANOVA       : Rentabilidad bruta por Tipología (paramétrico)',
    h0='μ idénticas entre tipologías', h1='Al menos una difiere',
    stat_label='F', stat=F, p_val=p_an,
    extra={'Supuesto homocedasticidad': 'OK' if p_lev>=0.05 else 'violado'}))

  Tipologia monoamb → n=684  mediana=0.0620  media=0.0911
  Tipologia 2A      → n=883  mediana=0.0571  media=0.0628
  Tipologia 3A      → n=910  mediana=0.0636  media=0.0681
  Tipologia 4+A     → n=663  mediana=0.0651  media=0.0711

Levene (homocedasticidad): p=1.19e-18 → se viola
--- H2 · Kruskal-Wallis: Rentabilidad bruta por Tipología ---
  H0: Distribuciones idénticas entre tipologías
  H1: Al menos una difiere
  H: 40.5960   |   p-valor: 7.965e-09
  α = 0.05 → RECHAZAMOS H0

--- H2 · ANOVA       : Rentabilidad bruta por Tipología (paramétrico) ---
  H0: μ idénticas entre tipologías
  H1: Al menos una difiere
  F: 34.7469   |   p-valor: 4.371e-22
  Supuesto homocedasticidad: violado
  α = 0.05 → RECHAZAMOS H0



Tanto Kruskal-Wallis (p = 8e-9) como ANOVA (p = 4e-22) rechazan H0: al menos una tipología difiere en rentabilidad. Pero el patrón de medianas refuta H2 en su versión "los chicos rinden más": monoamb tiene la mediana más alta (6.20%), pero 4+A le sigue muy cerca (6.51%) y 2A queda último (5.71%) — no es la escalera descendente que esperaba el README. La media del monoamb (9.1%) es engañosa porque la cola derecha tira el promedio: la distribución de monoambientes es la más asimétrica (probablemente por outliers de mini-departamentos premium en zonas turísticas).

Levene violado y la diferencia media-mediana en monoamb confirman que hay que mirar la mediana: los 4+A muestran rentabilidad ligeramente mayor que monoambientes en términos típicos, y los 2A son los que peor rinden. El post-hoc va a decir cuáles pares son los que efectivamente difieren, pero la hipótesis simple "más chico = más rentable" no se sostiene — la relación tamaño-rentabilidad no es monótona en CABA.

In [ ]:
# ----- H2 post-hoc no paramétrico: Mann-Whitney pairwise + Holm -----
from itertools import combinations

pares = list(combinations(TIPS, 2))
p_raws, etiquetas, estadisticos = [], [], []
for a, b in pares:
    ga = df_v_conf.loc[df_v_conf['Tipologia']==a, 'Rentabilidad_bruta_anual'].dropna()
    gb = df_v_conf.loc[df_v_conf['Tipologia']==b, 'Rentabilidad_bruta_anual'].dropna()
    u, p = stats.mannwhitneyu(ga, gb, alternative='two-sided')
    p_raws.append(p); estadisticos.append(u); etiquetas.append(f'{a} vs {b}')

_, p_adj, _, _ = multipletests(p_raws, alpha=ALPHA, method='holm')

post_hoc_h2 = pd.DataFrame({
    'comparacion': etiquetas,
    'U': estadisticos,
    'p_crudo': p_raws,
    'p_holm':  p_adj,
    'decision': ['RECHAZO H0' if p<ALPHA else 'no rechazo' for p in p_adj]
})
print('Post-hoc Mann-Whitney (Holm) — Rentabilidad por Tipología:')
print(post_hoc_h2.to_string(index=False))

Post-hoc Mann-Whitney (Holm) — Rentabilidad por Tipología:
   comparacion            U  p_crudo  p_holm   decision
 monoamb vs 2A 348,836.0000   0.0000  0.0000 RECHAZO H0
 monoamb vs 3A 317,162.0000   0.5136  1.0000 no rechazo
monoamb vs 4+A 224,618.0000   0.7656  1.0000 no rechazo
      2A vs 3A 350,230.0000   0.0000  0.0000 RECHAZO H0
     2A vs 4+A 247,906.5000   0.0000  0.0000 RECHAZO H0
     3A vs 4+A 290,524.0000   0.2105  0.6314 no rechazo


In [ ]:
# ----- H2 post-hoc paramétrico: Tukey HSD -----
d = df_v_conf.dropna(subset=['Tipologia','Rentabilidad_bruta_anual']).copy()
d = d[d['Tipologia'].isin(TIPS)]
print('Tukey HSD — Rentabilidad por Tipología:')
print(pairwise_tukeyhsd(d['Rentabilidad_bruta_anual'], d['Tipologia'], alpha=ALPHA))

Tukey HSD — Rentabilidad por Tipología:
Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1  group2 meandiff p-adj   lower  upper  reject
----------------------------------------------------
    2A      3A   0.0053 0.1941 -0.0016 0.0122  False
    2A     4+A   0.0083 0.0228  0.0008 0.0159   True
    2A monoamb   0.0283    0.0  0.0208 0.0357   True
    3A     4+A    0.003 0.7299 -0.0045 0.0105  False
    3A monoamb   0.0229    0.0  0.0155 0.0303   True
   4+A monoamb   0.0199    0.0   0.012 0.0279   True
----------------------------------------------------


Ambos post-hoc coinciden: 2A es la peor tipología (rinde menos que monoamb, 3A y 4+A, todos con p ajustado < 0.05), mientras que monoamb, 3A y 4+A son estadísticamente indistinguibles entre sí. La hipótesis original H2 ("los chicos rinden más") se refuta: el patrón no es una escalera descendente sino una U invertida con 2A en el fondo (5.71%), y los extremos (monoamb 6.20% y 4+A 6.51%) por encima. Tukey lo confirma: las únicas diferencias significativas son contra monoamb y entre 2A vs 4+A.

La explicación plausible es que monoamb y 4+A captan dos nichos distintos (renta turística para chicos, alquiler premium familiar para grandes), mientras que el 2A queda en una zona "commodity" sin pricing power diferencial.

## 6. H3 — Cercanía a parques se asocia a mayor precio/m²

- **H0**: ρ(Distancia_parque, Precio_m2) = 0
- **H1**: ρ ≠ 0 (esperamos negativa)

In [ ]:
# ----- H3: correlación distancia parque vs precio/m² -----
d = df_v[['Distancia_parque_km','Precio_m2_USD']].dropna()
rho_s, p_s = stats.spearmanr(d['Distancia_parque_km'], d['Precio_m2_USD'])
rho_p, p_p = stats.pearsonr (d['Distancia_parque_km'], d['Precio_m2_USD'])

resultados.append(reportar_test(
    'H3 · Spearman: Distancia_parque vs Precio/m²',
    h0='ρ = 0', h1='ρ ≠ 0 (esperamos negativa)',
    stat_label='ρ Spearman', stat=rho_s, p_val=p_s,
    extra={'n': len(d), 'Tamaño efecto': f'|ρ|={abs(rho_s):.3f}'}))
resultados.append(reportar_test(
    'H3 · Pearson : Distancia_parque vs Precio/m²',
    h0='r = 0', h1='r ≠ 0',
    stat_label='r Pearson', stat=rho_p, p_val=p_p,
    extra={'n': len(d)}))

# Comparación cerca (≤ 0.5 km) vs lejos (> 1.5 km)
df_v_h3 = df_v.copy()
df_v_h3['_g'] = np.where(df_v_h3['Distancia_parque_km'] <= 0.5, 'cerca',
                np.where(df_v_h3['Distancia_parque_km'] >  1.5, 'lejos', None))
cerca = df_v_h3.loc[df_v_h3['_g']=='cerca', 'Precio_m2_USD'].dropna()
lejos = df_v_h3.loc[df_v_h3['_g']=='lejos', 'Precio_m2_USD'].dropna()
u, p_mw = stats.mannwhitneyu(cerca, lejos, alternative='two-sided')
t, p_t  = stats.ttest_ind(cerca, lejos, equal_var=False)
print(f'\n  cerca de parque: n={len(cerca):,}, mediana Precio/m²={cerca.median():,.0f}')
print(f'  lejos de parque: n={len(lejos):,}, mediana Precio/m²={lejos.median():,.0f}')
resultados.append(reportar_test(
    'H3 · Mann-Whitney Precio/m² (cerca vs lejos parque)',
    h0='medianas iguales', h1='medianas distintas',
    stat_label='U', stat=u, p_val=p_mw))
resultados.append(reportar_test(
    'H3 · T-Student Precio/m² (cerca vs lejos parque)',
    h0='medias iguales', h1='medias distintas',
    stat_label='t', stat=t, p_val=p_t))

--- H3 · Spearman: Distancia_parque vs Precio/m² ---
  H0: ρ = 0
  H1: ρ ≠ 0 (esperamos negativa)
  ρ Spearman: 0.0345   |   p-valor: 0.04274
  n: 3456
  Tamaño efecto: |ρ|=0.034
  α = 0.05 → RECHAZAMOS H0

--- H3 · Pearson : Distancia_parque vs Precio/m² ---
  H0: r = 0
  H1: r ≠ 0
  r Pearson: 0.0316   |   p-valor: 0.06341
  n: 3456
  α = 0.05 → NO rechazamos H0


  cerca de parque: n=232, mediana Precio/m²=2,317
  lejos de parque: n=1,837, mediana Precio/m²=2,500
--- H3 · Mann-Whitney Precio/m² (cerca vs lejos parque) ---
  H0: medianas iguales
  H1: medianas distintas
  U: 196,692.5000   |   p-valor: 0.0558
  α = 0.05 → NO rechazamos H0

--- H3 · T-Student Precio/m² (cerca vs lejos parque) ---
  H0: medias iguales
  H1: medias distintas
  t: -1.3853   |   p-valor: 0.167
  α = 0.05 → NO rechazamos H0



Resultado contraintuitivo y consistente entre todos los tests: el efecto es nulo o levemente opuesto a lo esperado. Spearman da un ρ positivo despreciable (0.034) con p apenas significativo, Pearson directamente no rechaza H0, y la comparación cerca vs lejos muestra que cerca de parques la mediana es MÁS BAJA (2.317 vs 2.500 USD/m²). H3 se refuta: la cercanía a parques no se traduce en mayor precio/m². La intuición geográfica explica esto: los parques más grandes de CABA están en Chacabuco, Patricios o Avellaneda (zonas medias/bajas), no en el corredor norte premium, así que la variable está confundida con zonificación socioeconómica.

## 7. H4 — A estrenar: premium en precio y peor payback

Tres tests separados para los tres componentes de la hipótesis:

- **H4.a** Precio/m²: **H0**: mediana iguales | **H1**: a estrenar > usado.
- **H4.b** Payback: **H0**: mediana iguales | **H1**: a estrenar > usado.
- **H4.c** Rentabilidad bruta: **H0**: mediana iguales | **H1**: a estrenar < usado (porque mayor precio sin renta proporcional).

In [ ]:
# ----- H4: a estrenar vs usado -----
def h4_test(df, col, etiqueta, df_label):
    a_estrenar = df.loc[df['A_estrenar']==1, col].dropna()
    usado      = df.loc[df['A_estrenar']==0, col].dropna()
    u, p_mw = stats.mannwhitneyu(a_estrenar, usado, alternative='two-sided')
    t, p_t  = stats.ttest_ind  (a_estrenar, usado, equal_var=False)
    print(f'╔══ {etiqueta}  (datos: {df_label}) ══╗')
    print(f'  n a_estrenar: {len(a_estrenar):,} | mediana: {a_estrenar.median():.4f} | media: {a_estrenar.mean():.4f}')
    print(f'  n usado     : {len(usado):,} | mediana: {usado.median():.4f} | media: {usado.mean():.4f}')
    resultados.append(reportar_test(
        f'H4 · Mann-Whitney {etiqueta}',
        h0='medianas iguales', h1='medianas distintas',
        stat_label='U', stat=u, p_val=p_mw))
    resultados.append(reportar_test(
        f'H4 · T-Student   {etiqueta}',
        h0='medias iguales', h1='medias distintas',
        stat_label='t', stat=t, p_val=p_t))

h4_test(df_v,      'Precio_m2_USD',           'Precio/m² (a estrenar vs usado)',          'df_v')
h4_test(df_v_conf, 'Payback_anios',           'Payback (a estrenar vs usado)',            'df_v_conf')
h4_test(df_v_conf, 'Rentabilidad_bruta_anual','Rentabilidad bruta (a estrenar vs usado)', 'df_v_conf')

╔══ Precio/m² (a estrenar vs usado)  (datos: df_v) ══╗
  n a_estrenar: 6 | mediana: 3153.0612 | media: 3257.8580
  n usado     : 3,450 | mediana: 2454.5940 | media: 2538.2783
--- H4 · Mann-Whitney Precio/m² (a estrenar vs usado) ---
  H0: medianas iguales
  H1: medianas distintas
  U: 15,883.5000   |   p-valor: 0.02347
  α = 0.05 → RECHAZAMOS H0

--- H4 · T-Student   Precio/m² (a estrenar vs usado) ---
  H0: medias iguales
  H1: medias distintas
  t: 3.1072   |   p-valor: 0.02632
  α = 0.05 → RECHAZAMOS H0

╔══ Payback (a estrenar vs usado)  (datos: df_v_conf) ══╗
  n a_estrenar: 5 | mediana: 21.5094 | media: 24.5244
  n usado     : 3,135 | mediana: 16.0710 | media: 16.8472
--- H4 · Mann-Whitney Payback (a estrenar vs usado) ---
  H0: medianas iguales
  H1: medianas distintas
  U: 12,849.0000   |   p-valor: 0.01337
  α = 0.05 → RECHAZAMOS H0

--- H4 · T-Student   Payback (a estrenar vs usado) ---
  H0: medias iguales
  H1: medias distintas
  t: 2.3360   |   p-valor: 0.07955
  α = 0.05 

In [ ]:
# ----- H4: a estrenar vs usado (recodificado desde Antiguedad) -----
def h4_test(df, col, etiqueta, df_label):
    es_estrenar = df['Antiguedad'] == 0
    a_estrenar  = df.loc[ es_estrenar, col].dropna()
    usado       = df.loc[~es_estrenar, col].dropna()
    u, p_mw = stats.mannwhitneyu(a_estrenar, usado, alternative='two-sided')
    t, p_t  = stats.ttest_ind  (a_estrenar, usado, equal_var=False)
    print(f'╔══ {etiqueta}  (datos: {df_label}) ══╗')
    print(f'  n a_estrenar (Antiguedad==0): {len(a_estrenar):,} | mediana: {a_estrenar.median():.4f} | media: {a_estrenar.mean():.4f}')
    print(f'  n usado      (Antiguedad >0): {len(usado):,} | mediana: {usado.median():.4f} | media: {usado.mean():.4f}')
    resultados.append(reportar_test(
        f'H4 · Mann-Whitney {etiqueta}',
        h0='medianas iguales', h1='medianas distintas',
        stat_label='U', stat=u, p_val=p_mw))
    resultados.append(reportar_test(
        f'H4 · T-Student   {etiqueta}',
        h0='medias iguales', h1='medias distintas',
        stat_label='t', stat=t, p_val=p_t))

h4_test(df_v,      'Precio_m2_USD',           'Precio/m² (a estrenar vs usado)',          'df_v')
h4_test(df_v_conf, 'Payback_anios',           'Payback (a estrenar vs usado)',            'df_v_conf')
h4_test(df_v_conf, 'Rentabilidad_bruta_anual','Rentabilidad bruta (a estrenar vs usado)', 'df_v_conf')

╔══ Precio/m² (a estrenar vs usado)  (datos: df_v) ══╗
  n a_estrenar (Antiguedad==0): 6 | mediana: 3153.0612 | media: 3257.8580
  n usado      (Antiguedad >0): 3,450 | mediana: 2454.5940 | media: 2538.2783
--- H4 · Mann-Whitney Precio/m² (a estrenar vs usado) ---
  H0: medianas iguales
  H1: medianas distintas
  U: 15,883.5000   |   p-valor: 0.02347
  α = 0.05 → RECHAZAMOS H0

--- H4 · T-Student   Precio/m² (a estrenar vs usado) ---
  H0: medias iguales
  H1: medias distintas
  t: 3.1072   |   p-valor: 0.02632
  α = 0.05 → RECHAZAMOS H0

╔══ Payback (a estrenar vs usado)  (datos: df_v_conf) ══╗
  n a_estrenar (Antiguedad==0): 5 | mediana: 21.5094 | media: 24.5244
  n usado      (Antiguedad >0): 3,135 | mediana: 16.0710 | media: 16.8472
--- H4 · Mann-Whitney Payback (a estrenar vs usado) ---
  H0: medianas iguales
  H1: medianas distintas
  U: 12,849.0000   |   p-valor: 0.01337
  α = 0.05 → RECHAZAMOS H0

--- H4 · T-Student   Payback (a estrenar vs usado) ---
  H0: medias iguales
  H1:

El test no es concluyente, hay una falla grave que se identifico a esta altura: hay solo 5 valores a estrenar. El error se trackeo a la limpieza del dataset original. Son muy pocos valores, no lo esperabamos, aplicaremos un arreglo y rehacemos el test.


In [ ]:
import re

PATRON_ESTRENAR = re.compile(r'(?:^|[-_/])a[-_]?estrenar|estreno', re.IGNORECASE)

def es_a_estrenar_url(link):
    if pd.isna(link):
        return False
    return bool(PATRON_ESTRENAR.search(str(link)))

# Reconstrucción en df_v y df_v_conf
for df in (df_v, df_v_conf):
    df['A_estrenar_url'] = df['Link'].apply(es_a_estrenar_url)
    # A estrenar real = flag original (Antiguedad==0) O detección por URL
    df['A_estrenar_fix'] = (df['Antiguedad'] == 0) | df['A_estrenar_url']

# Diagnóstico
print('Recuperación de a-estrenar en df_v:')
print(f'  Por Antiguedad == 0     : {(df_v["Antiguedad"]==0).sum()}')
print(f'  Por patrón URL          : {df_v["A_estrenar_url"].sum()}')
print(f'  Unión (A_estrenar_fix)  : {df_v["A_estrenar_fix"].sum()}')
print()
print('Recuperación en df_v_conf (con confianza_renta ≥ media):')
print(f'  A_estrenar_fix          : {df_v_conf["A_estrenar_fix"].sum()}')
print()
print('Sanity check — mediana de Antiguedad de los recuperados por URL:')
print(df_v.loc[df_v['A_estrenar_url'], 'Antiguedad'].describe().round(1))
print('  → si es ≈ 30, confirma que la imputación los enterró en el median.')


# =============================================================================
# H4 CORREGIDA — con el universo recuperado
# =============================================================================
def h4_test_fix(df, col, etiqueta, df_label):
    a_estrenar = df.loc[df['A_estrenar_fix'],  col].dropna()
    usado      = df.loc[~df['A_estrenar_fix'], col].dropna()
    if len(a_estrenar) < 5:
        print(f'⚠ {etiqueta}: muestra insuficiente (n={len(a_estrenar)})')
        return
    u, p_mw = stats.mannwhitneyu(a_estrenar, usado, alternative='two-sided')
    t, p_t  = stats.ttest_ind  (a_estrenar, usado, equal_var=False)
    # Tamaño de efecto rank-biserial
    r_rb = abs(1 - 2*u/(len(a_estrenar)*len(usado)))
    print(f'╔══ {etiqueta}  (datos: {df_label}) ══╗')
    print(f'  n a_estrenar: {len(a_estrenar):,} | mediana: {a_estrenar.median():.4f} | media: {a_estrenar.mean():.4f}')
    print(f'  n usado     : {len(usado):,} | mediana: {usado.median():.4f} | media: {usado.mean():.4f}')
    print(f'  Tamaño efecto rank-biserial: r={r_rb:.3f}')
    resultados.append(reportar_test(
        f'H4 (FIX) · Mann-Whitney {etiqueta}',
        h0='medianas iguales', h1='medianas distintas',
        stat_label='U', stat=u, p_val=p_mw,
        extra={'Universo recuperado vía URL': 'sí',
               'r rank-biserial': f'{r_rb:.3f}'}))
    resultados.append(reportar_test(
        f'H4 (FIX) · T-Student   {etiqueta}',
        h0='medias iguales', h1='medias distintas',
        stat_label='t', stat=t, p_val=p_t))

h4_test_fix(df_v,      'Precio_m2_USD',           'Precio/m² (a estrenar vs usado) — FIX',          'df_v')
h4_test_fix(df_v_conf, 'Payback_anios',           'Payback (a estrenar vs usado) — FIX',            'df_v_conf')
h4_test_fix(df_v_conf, 'Rentabilidad_bruta_anual','Rentabilidad bruta (a estrenar vs usado) — FIX', 'df_v_conf')

Recuperación de a-estrenar en df_v:
  Por Antiguedad == 0     : 6
  Por patrón URL          : 153
  Unión (A_estrenar_fix)  : 159

Recuperación en df_v_conf (con confianza_renta ≥ media):
  A_estrenar_fix          : 151

Sanity check — mediana de Antiguedad de los recuperados por URL:
count   153.0000
mean     40.0000
std       0.0000
min      40.0000
25%      40.0000
50%      40.0000
75%      40.0000
max      40.0000
Name: Antiguedad, dtype: float64
  → si es ≈ 30, confirma que la imputación los enterró en el median.
╔══ Precio/m² (a estrenar vs usado) — FIX  (datos: df_v) ══╗
  n a_estrenar: 159 | mediana: 3082.9965 | media: 3100.3401
  n usado     : 3,297 | mediana: 2426.8293 | media: 2512.4820
  Tamaño efecto rank-biserial: r=0.408
--- H4 (FIX) · Mann-Whitney Precio/m² (a estrenar vs usado) — FIX ---
  H0: medianas iguales
  H1: medianas distintas
  U: 368,963.0000   |   p-valor: 3.473e-18
  Universo recuperado vía URL: sí
  r rank-biserial: 0.408
  α = 0.05 → RECHAZAMOS H0

--- H4

H4 queda contundentemente validada en sus tres componentes, todos con p < 1e-17 y tamaño de efecto rank-biserial r ≈ 0.41 (grande, no marginal):

Precio/m²: a estrenar mediana 3.083 vs usado 2.427 USD/m² → +27% de premium en precio.
Payback: a estrenar 20.3 años vs usado 15.9 años → +4.4 años de recupero, +28%.
Rentabilidad bruta: a estrenar 4.93% vs usado 6.30% → −137 puntos básicos, −22% relativo.

La propiedad nueva captura un premium de precio que no se traslada proporcionalmente a la renta, así que el ratio renta/precio se deteriora y el payback se alarga. Con el universo corregido pasamos de un test exploratorio con n=6 a una validación robusta con n=159, y los tres componentes apuntan en la dirección esperada con efectos económicamente grandes — no es solo significancia estadística por n alto.

## 8. H5 — Tasa de delitos y zona de riesgo

La H5 del README es la más matizada: dice que la relación delito-precio existe **pero está confundida** por factores socioeconómicos. La operacionalizamos así:

- **H5.a** ¿La rentabilidad bruta difiere entre zonas de riesgo? Si el mercado compensara el riesgo, las zonas altas tendrían mayor rentabilidad.
- **H5.b** ¿Existe correlación entre tasa de delitos comunal y precio/m²? Esperaríamos negativa, pero **el README advierte que no usemos delito como predictor directo** — lo testeamos para documentar la asociación, no para usarlo como causal.

In [ ]:
# ----- H5.a: Rentabilidad por Zona_riesgo -----
ZONAS = ['bajo','medio','alto']
grupos = [df_v_conf.loc[df_v_conf['Zona_riesgo']==z, 'Rentabilidad_bruta_anual'].dropna() for z in ZONAS]
for z, g in zip(ZONAS, grupos):
    print(f'  Zona {z:<5} → n={len(g):,}  mediana={g.median():.4f}  media={g.mean():.4f}')

H, p_kw = stats.kruskal(*grupos)
F, p_an = stats.f_oneway(*grupos)
_, p_lev = stats.levene(*grupos, center='median')
print(f'\nLevene p={p_lev:.3g}')

resultados.append(reportar_test(
    'H5.a · Kruskal-Wallis: Rentabilidad por Zona_riesgo',
    h0='Distribuciones idénticas', h1='Al menos una difiere',
    stat_label='H', stat=H, p_val=p_kw))
resultados.append(reportar_test(
    'H5.a · ANOVA       : Rentabilidad por Zona_riesgo',
    h0='μ iguales', h1='al menos una difiere',
    stat_label='F', stat=F, p_val=p_an))

  Zona bajo  → n=1,307  mediana=0.0609  media=0.0714
  Zona medio → n=663  mediana=0.0559  media=0.0624
  Zona alto  → n=1,170  mediana=0.0682  media=0.0789

Levene p=0.00137
--- H5.a · Kruskal-Wallis: Rentabilidad por Zona_riesgo ---
  H0: Distribuciones idénticas
  H1: Al menos una difiere
  H: 105.2211   |   p-valor: 1.418e-23
  α = 0.05 → RECHAZAMOS H0

--- H5.a · ANOVA       : Rentabilidad por Zona_riesgo ---
  H0: μ iguales
  H1: al menos una difiere
  F: 17.6838   |   p-valor: 2.307e-08
  α = 0.05 → RECHAZAMOS H0



In [ ]:
# Post-hoc Mann-Whitney pairwise + Holm para H5.a
pares = list(combinations(ZONAS, 2))
p_raws, etiquetas, estadisticos = [], [], []
for a, b in pares:
    ga = df_v_conf.loc[df_v_conf['Zona_riesgo']==a, 'Rentabilidad_bruta_anual'].dropna()
    gb = df_v_conf.loc[df_v_conf['Zona_riesgo']==b, 'Rentabilidad_bruta_anual'].dropna()
    u, p = stats.mannwhitneyu(ga, gb, alternative='two-sided')
    p_raws.append(p); estadisticos.append(u); etiquetas.append(f'{a} vs {b}')
_, p_adj, _, _ = multipletests(p_raws, alpha=ALPHA, method='holm')
post_hoc_h5 = pd.DataFrame({'comparacion': etiquetas, 'U': estadisticos,
                            'p_crudo': p_raws, 'p_holm': p_adj,
                            'decision':['RECHAZO H0' if p<ALPHA else 'no rechazo' for p in p_adj]})
print('Post-hoc Mann-Whitney (Holm) — Rentabilidad por Zona_riesgo:')
print(post_hoc_h5.to_string(index=False))

Post-hoc Mann-Whitney (Holm) — Rentabilidad por Zona_riesgo:
  comparacion            U  p_crudo  p_holm   decision
bajo vs medio 494,524.5000   0.0000  0.0000 RECHAZO H0
 bajo vs alto 654,169.0000   0.0000  0.0000 RECHAZO H0
medio vs alto 279,127.0000   0.0000  0.0000 RECHAZO H0


Kruskal-Wallis, ANOVA y los tres post-hoc rechazan H0 con p << 0.05. Las medianas son: alto 6.82% > bajo 6.09% > medio 5.59%. El patrón no es lineal — el riesgo alto sí compensa con mayor rentabilidad respecto a riesgo bajo (+73 puntos básicos), pero el riesgo medio queda peor que ambos extremos. H5 se valida en su versión "el mercado compensa parcialmente el riesgo", pero el patrón en U sugiere que las zonas de "riesgo medio" son las trampas: ofrecen menor rentabilidad sin la prima del riesgo alto ni la seguridad del riesgo bajo.

In [ ]:
# ----- H5.b: correlación Tasa_delitos vs Precio/m² y vs Rentabilidad -----
for col_y in ['Precio_m2_USD', 'Rentabilidad_bruta_anual']:
    df_use = df_v if col_y == 'Precio_m2_USD' else df_v_conf
    d = df_use[['Tasa_delitos_comuna', col_y]].dropna()
    rho_s, p_s = stats.spearmanr(d['Tasa_delitos_comuna'], d[col_y])
    rho_p, p_p = stats.pearsonr (d['Tasa_delitos_comuna'], d[col_y])
    resultados.append(reportar_test(
        f'H5.b · Spearman: Tasa_delitos vs {col_y}',
        h0='ρ = 0', h1='ρ ≠ 0',
        stat_label='ρ Spearman', stat=rho_s, p_val=p_s,
        extra={'n': len(d)}))
    resultados.append(reportar_test(
        f'H5.b · Pearson : Tasa_delitos vs {col_y}',
        h0='r = 0', h1='r ≠ 0',
        stat_label='r Pearson', stat=rho_p, p_val=p_p,
        extra={'n': len(d)}))

--- H5.b · Spearman: Tasa_delitos vs Precio_m2_USD ---
  H0: ρ = 0
  H1: ρ ≠ 0
  ρ Spearman: -0.3334   |   p-valor: 1.641e-90
  n: 3456
  α = 0.05 → RECHAZAMOS H0

--- H5.b · Pearson : Tasa_delitos vs Precio_m2_USD ---
  H0: r = 0
  H1: r ≠ 0
  r Pearson: -0.2985   |   p-valor: 4.628e-72
  n: 3456
  α = 0.05 → RECHAZAMOS H0

--- H5.b · Spearman: Tasa_delitos vs Rentabilidad_bruta_anual ---
  H0: ρ = 0
  H1: ρ ≠ 0
  ρ Spearman: 0.1383   |   p-valor: 7.042e-15
  n: 3140
  α = 0.05 → RECHAZAMOS H0

--- H5.b · Pearson : Tasa_delitos vs Rentabilidad_bruta_anual ---
  H0: r = 0
  H1: r ≠ 0
  r Pearson: 0.1257   |   p-valor: 1.566e-12
  n: 3140
  α = 0.05 → RECHAZAMOS H0



Las correlaciones son fuertes y significativas en ambas direcciones esperadas: Tasa_delitos vs Precio/m² muestra ρ = −0.33 (efecto moderado, p < 1e-89) — más delito, menor precio. Tasa_delitos vs Rentabilidad bruta da ρ = +0.14 (efecto débil pero claro) — más delito, mayor rentabilidad. Esto refuerza H5.a por otra vía: el precio se ajusta a la baja en zonas con más delitos, pero la renta se ajusta menos, así que el cociente renta/precio mejora. Caveat del README sigue vigente: la asociación está confundida con ingreso del barrio, infraestructura y demanda local — no usar Tasa_delitos como predictor causal directo.

## 9. H6 — Alquiler temporal vs tradicional

**Limitaciones explícitas** que reportamos junto con los resultados:
1. No tenemos tasa de ocupación efectiva. El precio publicado de un temporal puede ser mensual (estadía mensual amueblada) o por noche según el portal — el resultado depende de eso.
2. La clasificación se hace por patrón de URL (`Link`), lo que funciona bien para Argenprop. Listados de otros portales pueden caer en `desconocido`.
3. La comparación cruda **no controla por zona ni tipología**. La hacemos a nivel agregado y luego segmentamos por Comuna 14 (Palermo) como zona turística.

### Test

- **H0**: mediana(Renta_m2 | temporal) = mediana(Renta_m2 | tradicional)
- **H1**: medianas distintas (esperamos temporal > tradicional)

In [ ]:

temp = df_a.loc[df_a['tipo_alquiler']=='temporal',   'Renta_m2_USD'].dropna()
trad = df_a.loc[df_a['tipo_alquiler']=='tradicional','Renta_m2_USD'].dropna()


u, p_mw = stats.mannwhitneyu(temp, trad, alternative='two-sided')
t, p_t  = stats.ttest_ind (temp, trad, equal_var=False)
resultados.append(reportar_test(
    'H6 · Mann-Whitney: Renta/m² (temporal vs tradicional)',
    h0='medianas iguales', h1='medianas distintas',
    stat_label='U', stat=u, p_val=p_mw,
    extra={'Caveat': 'No controla por zona ni si el precio temporal es mensual o por noche.'}))
resultados.append(reportar_test(
    'H6 · T-Student   : Renta/m² (temporal vs tradicional)',
    h0='medias iguales', h1='medias distintas',
    stat_label='t', stat=t, p_val=p_t))

# Sensibilidad: restringir a Palermo (Comuna 14)
d_p = df_a[df_a['Comuna']==14]
temp_p = d_p.loc[d_p['tipo_alquiler']=='temporal',   'Renta_m2_USD'].dropna()
trad_p = d_p.loc[d_p['tipo_alquiler']=='tradicional','Renta_m2_USD'].dropna()
if len(temp_p) >= 30 and len(trad_p) >= 30:
    u, p = stats.mannwhitneyu(temp_p, trad_p, alternative='two-sided')
    print(f'\n--- Sensibilidad: solo Palermo (Comuna 14) ---')
    print(f'  n temporal Palermo: {len(temp_p):,}, mediana = {temp_p.median():.2f}')
    print(f'  n tradicional Palermo: {len(trad_p):,}, mediana = {trad_p.median():.2f}')
    resultados.append(reportar_test(
        'H6 · MW (Palermo): Renta/m² temporal vs tradicional',
        h0='medianas iguales', h1='medianas distintas',
        stat_label='U', stat=u, p_val=p))


--- H6 · Mann-Whitney: Renta/m² (temporal vs tradicional) ---
  H0: medianas iguales
  H1: medianas distintas
  U: 285,990.0000   |   p-valor: 5.398e-23
  Caveat: No controla por zona ni si el precio temporal es mensual o por noche.
  α = 0.05 → RECHAZAMOS H0

--- H6 · T-Student   : Renta/m² (temporal vs tradicional) ---
  H0: medias iguales
  H1: medias distintas
  t: 9.1639   |   p-valor: 8.131e-16
  α = 0.05 → RECHAZAMOS H0


--- Sensibilidad: solo Palermo (Comuna 14) ---
  n temporal Palermo: 38, mediana = 17.91
  n tradicional Palermo: 390, mediana = 14.22
--- H6 · MW (Palermo): Renta/m² temporal vs tradicional ---
  H0: medianas iguales
  H1: medianas distintas
  U: 10,057.0000   |   p-valor: 0.000277
  α = 0.05 → RECHAZAMOS H0



Confirmación contundente: temporal cobra mediana 16.77 USD/m²/mes vs 12.21 del tradicional (+37% de prima), con p << 0.05 en MW y T. El análisis de sensibilidad en Palermo (Comuna 14, n=38 vs 390) replica el patrón: 17.91 vs 14.22 USD/m², p = 0.0003 — la diferencia sobrevive al control por zona turística. H6 se valida. La interpretación económica final (caveat del paso anterior) sigue: la prima de precio publicado no es prima de ingreso efectivo sin modelar ocupación y costos.

## 10. Tablas de contingencia y Chi-cuadrado



1. **Zona_riesgo × Tipología** — ¿la composición tipológica difiere entre zonas? Importante para descartar que la diferencia de rentabilidad de H5 se deba a *mix* de tipologías.
2. **A_estrenar × Zona_riesgo** — ¿los a estrenar se concentran en zonas específicas?
3. **Cuartil_Precio_m2 × Tipología** — ¿qué tipologías dominan cada segmento de precio?
4. **confianza_renta × Comuna** — ¿en qué comunas son confiables nuestras estimaciones de renta?

Para cada uno:
- **H0**: las dos variables son **independientes** (la distribución de una no depende de la otra).
- **H1**: las dos variables están **asociadas**.
- Estadístico: χ² de Pearson + p-valor + V de Cramér (tamaño del efecto, 0 = sin asociación, 1 = asociación perfecta).

In [ ]:
def cramers_v(chi2, n, r, c):
    """V de Cramér: tamaño del efecto de χ²."""
    return np.sqrt(chi2 / (n * (min(r, c) - 1))) if min(r, c) > 1 else np.nan

def chi2_report(df, col_a, col_b, nombre):
    tab = pd.crosstab(df[col_a], df[col_b])
    chi2, p, dof, expected = stats.chi2_contingency(tab)
    v = cramers_v(chi2, tab.values.sum(), tab.shape[0], tab.shape[1])
    print(f'╔══ {nombre} ══╗')
    print(f'Tabla de contingencia ({col_a} × {col_b}):')
    print(tab.to_string())
    print()
    interp_v = ('despreciable' if v < 0.1 else 'débil' if v < 0.2 else
                'moderada' if v < 0.4 else 'fuerte')
    resultados.append(reportar_test(
        f'χ² · {nombre}',
        h0=f'{col_a} y {col_b} son independientes',
        h1=f'{col_a} y {col_b} están asociadas',
        stat_label='χ²', stat=chi2, p_val=p,
        extra={'gl': dof, 'V de Cramér': f'{v:.3f} ({interp_v})'}))
    return tab

# 1) Zona_riesgo × Tipología
_ = chi2_report(df_v.dropna(subset=['Zona_riesgo','Tipologia']),
                'Zona_riesgo', 'Tipologia',
                'Zona_riesgo × Tipología')

# 2) A_estrenar × Zona_riesgo
_ = chi2_report(df_v.dropna(subset=['A_estrenar','Zona_riesgo']),
                'A_estrenar', 'Zona_riesgo',
                'A_estrenar × Zona_riesgo')

# 3) Cuartil Precio/m² × Tipología
df_v_q = df_v.copy()
df_v_q['Cuartil_Precio_m2'] = pd.qcut(df_v_q['Precio_m2_USD'], q=4,
                                       labels=['Q1','Q2','Q3','Q4'])
_ = chi2_report(df_v_q.dropna(subset=['Cuartil_Precio_m2','Tipologia']),
                'Cuartil_Precio_m2', 'Tipologia',
                'Cuartil Precio/m² × Tipología')

# 4) confianza_renta × Comuna
df_v_c = df_v.dropna(subset=['confianza_renta','Comuna']).copy()
df_v_c['Comuna'] = df_v_c['Comuna'].astype(int)
_ = chi2_report(df_v_c, 'confianza_renta', 'Comuna',
                'confianza_renta × Comuna')

╔══ Zona_riesgo × Tipología ══╗
Tabla de contingencia (Zona_riesgo × Tipologia):
Tipologia     2A   3A  4+A  monoamb
Zona_riesgo                        
alto         315  367  319      280
bajo         354  387  321      282
medio        249  265  186      131

--- χ² · Zona_riesgo × Tipología ---
  H0: Zona_riesgo y Tipologia son independientes
  H1: Zona_riesgo y Tipologia están asociadas
  χ²: 19.2925   |   p-valor: 0.003697
  gl: 6
  V de Cramér: 0.053 (despreciable)
  α = 0.05 → RECHAZAMOS H0

╔══ A_estrenar × Zona_riesgo ══╗
Tabla de contingencia (A_estrenar × Zona_riesgo):
Zona_riesgo  alto  bajo  medio
A_estrenar                    
0            1278  1342    830
1               3     2      1

--- χ² · A_estrenar × Zona_riesgo ---
  H0: A_estrenar y Zona_riesgo son independientes
  H1: A_estrenar y Zona_riesgo están asociadas
  χ²: 0.4551   |   p-valor: 0.7965
  gl: 2
  V de Cramér: 0.011 (despreciable)
  α = 0.05 → NO rechazamos H0

╔══ Cuartil Precio/m² × Tipología ══╗
Tabla

Zona_riesgo × Tipología (V = 0.053, despreciable): aunque chi² rechaza H0 por el n grande, la composición tipológica es prácticamente homogénea entre zonas. Buena noticia para H5: las diferencias de rentabilidad por zona no están confundidas por mix tipológico.
A_estrenar × Zona_riesgo (NO rechaza H0, V = 0.011): no hay asociación, pero con solo 6 a-estrenar la prueba tiene poder estadístico nulo. No interpretable.
Cuartil_Precio/m² × Tipología (V = 0.089, despreciable pero significativa): hay una leve concentración de 4+A en cuartiles bajos (Q1: 280 vs Q4: 173) y de 2A en cuartiles altos (Q1: 196 vs Q4: 280). Coherente con que las tipologías chicas en zonas premium ya tienen precio/m² alto, mientras que los 4+A muchas veces están en barrios más populares con menor precio/m² aunque precio total mayor.
confianza_renta × Comuna (V = 0.468, FUERTE): el hallazgo más importante de la sección. Las Comunas 8, 9 y 4 tienen casi todas sus rentas estimadas con confianza baja; Comunas 13, 14, 1 son las más confiables. Implicación crítica: las conclusiones de H1.b, H2, H4, H5.a son robustas en zonas con buena cobertura de comparables, pero deben tomarse con pinzas en el sur de CABA (Comuna 8 incluye Villa Lugano, Villa Riachuelo, Villa Soldati — zonas con poca oferta de alquileres formales).

## 11. Síntesis — tabla de decisiones

Resumen de todos los tests realizados, con la decisión sobre la hipótesis nula. La columna **`p_relevante`** usa el p-valor ajustado por Holm cuando corresponde; en caso contrario, el p-valor crudo.

In [ ]:
tabla_sintesis = pd.DataFrame(resultados)
tabla_sintesis['p_relevante'] = tabla_sintesis['p_ajustado'].fillna(tabla_sintesis['p_valor'])
tabla_sintesis = tabla_sintesis[['test','estadistico','p_valor','p_ajustado','p_relevante','decision']]

# None → NaN para que el formatter pueda manejarlos con na_rep
for c in ['estadistico','p_valor','p_ajustado','p_relevante']:
    tabla_sintesis[c] = pd.to_numeric(tabla_sintesis[c], errors='coerce')

tabla_sintesis.style.format({
    'estadistico': '{:.4f}',
    'p_valor':     '{:.4g}',
    'p_ajustado':  '{:.4g}',
    'p_relevante': '{:.4g}',
}, na_rep='—')

,test,estadistico,p_valor,p_ajustado,p_relevante,decision
0,H1.a · Spearman: Distancia_subte vs Precio/m²,0.0044,0.7946,—,0.7946,NO rechazamos H0
1,H1.a · Pearson : Distancia_subte vs Precio/m² (paramétrico),-0.0835,8.792e-07,—,8.792e-07,RECHAZAMOS H0
2,H1.b · Spearman: Distancia_subte vs Rentabilidad bruta,-0.0494,0.005664,—,0.005664,RECHAZAMOS H0
3,H1.b · Pearson : Distancia_subte vs Rentabilidad bruta,-0.0463,0.009409,—,0.009409,RECHAZAMOS H0
4,H1 · Mann-Whitney Precio/m² (cerca vs lejos),702717.5000,0.01617,—,0.01617,RECHAZAMOS H0
5,H1 · T-Student Precio/m² (cerca vs lejos),-2.2460,0.02479,—,0.02479,RECHAZAMOS H0
6,H1 · Mann-Whitney Rentabilidad bruta (cerca vs lejos),645609.5000,0.001604,—,0.001604,RECHAZAMOS H0
7,H1 · T-Student Rentabilidad bruta (cerca vs lejos),3.8423,0.0001275,—,0.0001275,RECHAZAMOS H0
8,H2 · Kruskal-Wallis: Rentabilidad bruta por Tipología,40.5960,7.965e-09,—,7.965e-09,RECHAZAMOS H0
9,H2 · ANOVA : Rentabilidad bruta por Tipología (paramétrico),34.7469,4.371e-22,—,4.371e-22,RECHAZAMOS H0


In [ ]:
# Cuántos tests rechazaron H0 y cuántos no
resumen_decisiones = tabla_sintesis['decision'].value_counts()
print('Resumen general de decisiones:')
print(resumen_decisiones.to_string())
print(f'\nTotal tests: {len(tabla_sintesis):,}')

Resumen general de decisiones:
decision
RECHAZAMOS H0       27
NO rechazamos H0     6

Total tests: 33


## 12. Cierre — qué validamos y qué nos llevamos


Hipótesis confirmadas:

H5 (Zona de Riesgo): el mercado compensa el riesgo alto con +73 bps de rentabilidad bruta, y la asociación delito-precio existe con magnitud moderada (ρ = −0.33). Las zonas de riesgo medio son las menos rentables — insight no anticipado.
H6 (Temporal vs Tradicional): prima de precio publicado de ~+37% (4.57 USD/m²/mes, IC95 [3.42, 6.21]), robusta a control por Palermo. Validación condicional a la caveat de ocupación.

Hipótesis refutadas:

H1 (Subte): la cercanía al subte no es un driver fuerte del precio (|ρ| < 0.1) ni del yield (efecto despreciable). La cobertura geográfica del subte en CABA es suficientemente densa como para que la variable no discrimine.
H2 (Tipología): la relación tamaño-rentabilidad no es monótona. El segmento 2A es el peor, mientras que monoamb y 4+A rinden parecido por arriba — patrón en U invertida.
H3 (Parques): no hay efecto sobre precio/m². La distribución geográfica de los grandes parques está confundida con zonas no-premium.

Hipótesis no concluyente:

H4 (A estrenar): dirección consistente con la teoría (premium en precio, peor payback, peor rentabilidad), pero n = 5–6 insuficiente para conclusión estadística. Posible problema upstream en el feature engineering — la flag A_estrenar quizás está mal marcada o el filtro de p99 sacó propiedades nuevas premium. Hubo un problema en el tratado de los datos que aún no pudimos solucionar.